In [1]:
import platform, os
print(platform.system())
print(round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 1), "GB RAM")

Linux
13.6 GB RAM


In [2]:
! pip install -q --upgrade kaggle duckdb implicit

In [3]:
import os, getpass
from pathlib import Path

token = getpass.getpass("Paste your Kaggle token: ").strip()
print("token length:", len(token))

os.environ["KAGGLE_API_TOKEN"] = token
kdir = Path.home() / ".kaggle"
kdir.mkdir(exist_ok=True)
(kdir / "access_token").write_text(token)
os.chmod(kdir / "access_token", 0o600)

token length: 37


In [4]:
COMP = "h-and-m-personalized-fashion-recommendations"
for f in ["transactions_train.csv", "customers.csv", "articles.csv"]:
    !kaggle competitions download -c {COMP} -f {f} -p /content/raw -q
! cd /content/raw && unzip -o -q '*.zip' && rm -f *.zip
! ls -lh /content/raw


3 archives were successfully processed.
total 3.5G
-rw-r--r-- 1 root root  35M Jan 17  2022 articles.csv
-rw-r--r-- 1 root root 198M Jan 17  2022 customers.csv
-rw-r--r-- 1 root root 3.3G Jan 17  2022 transactions_train.csv


In [5]:
!rm -rf /content/repo
!git clone -q https://github.com/WoahImCoding/ecommerce-recommender.git /content/repo
!ls /content/repo/src

import sys
sys.path.append("/content/repo")
from src.metrics import evaluate
print("metrics imported")

metrics.py
metrics imported


In [6]:
import duckdb
from pathlib import Path

raw, proc = Path("/content/raw"), Path("/content/processed")
proc.mkdir(exist_ok=True)

con = duckdb.connect()
for name in ["transactions_train", "customers", "articles"]:
    con.execute(f"COPY (SELECT * FROM read_csv_auto('{raw}/{name}.csv')) TO '{proc}/{name}.parquet' (FORMAT PARQUET)")

con.execute(f"CREATE VIEW transactions AS SELECT * FROM '{proc}/transactions_train.parquet'")
con.execute(f"CREATE VIEW articles AS SELECT * FROM '{proc}/articles.parquet'")
con.execute("""
    CREATE OR REPLACE VIEW tx AS
    SELECT *, CAST(date_diff('day', t_dat, (SELECT max(t_dat) FROM transactions)) // 7 AS INTEGER) AS week_ago
    FROM transactions
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [9]:
# Give every customer a small integer ID
con.execute("""
    CREATE OR REPLACE TABLE cust_map AS
    SELECT customer_id,
           CAST(row_number() OVER (ORDER BY customer_id) - 1 AS INTEGER) AS cust
    FROM (SELECT DISTINCT customer_id FROM tx)
""")

# Load only the last 16 weeks, with compact column types
recent = con.sql("""
    SELECT m.cust,
           CAST(t.article_id AS INTEGER)       AS article_id,
           t.t_dat,
           CAST(t.week_ago AS SMALLINT)        AS week_ago,
           CAST(t.price AS FLOAT)              AS price,
           CAST(t.sales_channel_id AS TINYINT) AS channel
    FROM tx t
    JOIN cust_map m USING (customer_id)
    WHERE t.week_ago <= 15
""").df()

print(f"{len(recent):,} rows, {recent.memory_usage(deep=True).sum() / 1e6:.0f} MB")
recent.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

5,063,720 rows, 116 MB


,cust,article_id,t_dat,week_ago,price,channel
0,485302,820508001,2020-06-04,15,0.021593,2
1,485302,490176004,2020-06-04,15,0.021593,2
2,485417,658030017,2020-06-04,15,0.050831,2
3,485417,658030017,2020-06-04,15,0.050831,2
4,485417,740720002,2020-06-04,15,0.016932,2


In [10]:
cust_feat = con.sql("""
    SELECT m.cust,
           CAST(c.age AS DOUBLE)                 AS age,
           COALESCE(CAST(c.FN AS DOUBLE), 0)     AS fn,
           COALESCE(CAST(c.Active AS DOUBLE), 0) AS active,
           CASE c.club_member_status WHEN 'ACTIVE' THEN 0 WHEN 'PRE-CREATE' THEN 1
                                     WHEN 'LEFT CLUB' THEN 2 ELSE 3 END AS club,
           CASE c.fashion_news_frequency WHEN 'Regularly' THEN 2 WHEN 'Monthly' THEN 1 ELSE 0 END AS news
    FROM '/content/processed/customers.parquet' c
    JOIN cust_map m USING (customer_id)
""").df().set_index("cust").astype("float32")

art = con.sql("""
    SELECT CAST(article_id AS INTEGER)        AS article_id,
           CAST(product_type_no AS INTEGER)   AS product_type_no,
           CAST(colour_group_code AS INTEGER) AS colour_group_code,
           CAST(department_no AS INTEGER)     AS department_no,
           product_group_name, index_code
    FROM articles
""").df()
art["product_group_code"] = art["product_group_name"].astype("category").cat.codes
art["index_code_num"] = art["index_code"].astype("category").cat.codes
art = art.set_index("article_id")[["product_type_no", "colour_group_code", "department_no",
                                   "product_group_code", "index_code_num"]]
print(len(cust_feat), "customers,", len(art), "articles")

1362281 customers, 105542 articles


In [11]:
def add_features(cands, hist, cust_feat, art):
    """Add user, item and user-item features, computed only from `hist` (the weeks before the label week)."""
    ref = hist["t_dat"].max() + pd.Timedelta(days=1)
    days_ago = (ref - hist["t_dat"]).dt.days

    # User features: how this customer shops
    g = hist.groupby("cust")
    user = pd.DataFrame({
        "u_n_tx": g.size(),
        "u_n_days": g["t_dat"].nunique(),
        "u_days_since_last": days_ago.groupby(hist["cust"]).min(),
        "u_avg_price": g["price"].mean(),
        "u_online_share": (hist["channel"] == 2).groupby(hist["cust"]).mean(),
    })

    # Item features: how this item is selling
    gi = hist.groupby("article_id")
    item = pd.DataFrame({
        "i_cnt_1w": hist[days_ago <= 7].groupby("article_id").size(),
        "i_cnt_4w": hist[days_ago <= 28].groupby("article_id").size(),
        "i_cnt_12w": gi.size(),
        "i_n_buyers": gi["cust"].nunique(),
        "i_avg_price": gi["price"].mean(),
        "i_days_since_first": days_ago.groupby(hist["article_id"]).max(),
        "i_mean_buyer_age": hist.assign(age=hist["cust"].map(cust_feat["age"])).groupby("article_id")["age"].mean(),
    })
    item[["i_cnt_1w", "i_cnt_4w"]] = item[["i_cnt_1w", "i_cnt_4w"]].fillna(0)
    item["i_trend"] = item["i_cnt_1w"] / (item["i_cnt_4w"] / 4 + 1)

    out = (cands.join(user, on="cust")
                .join(cust_feat, on="cust")
                .join(item, on="article_id")
                .join(art, on="article_id"))

    # User-item features: does this item fit this customer?
    hg = hist[["cust", "article_id"]].join(art["product_group_code"], on="article_id")
    share = (hg.groupby(["cust", "product_group_code"]).size()
               .div(hg.groupby("cust").size(), level="cust")
               .rename("ui_group_share").reset_index())
    out = out.merge(share, on=["cust", "product_group_code"], how="left")
    out["ui_group_share"] = out["ui_group_share"].fillna(0)
    out["ui_price_ratio"] = out["i_avg_price"] / out["u_avg_price"]
    out["ui_age_gap"] = (out["age"] - out["i_mean_buyer_age"]).abs()

    floats = out.select_dtypes("float64").columns
    out[floats] = out[floats].astype("float32")
    return out

The Trained Data from my Kaggle Dataset

In [12]:
KAGGLE_USER = "aparnarameshdatas"          # <- change this
ART = "/content/kaggle_artifacts"

!mkdir -p {ART}
!kaggle datasets download {KAGGLE_USER}/hm-recommender-artifacts -p {ART} --unzip
!ls -lh {ART}

Dataset URL: https://www.kaggle.com/datasets/aparnarameshdatas/hm-recommender-artifacts
License(s): CC0-1.0
100% 909k/909k [00:00<00:00, 85.9MB/s]

total 2.1M
-rw-r--r-- 1 root root 2.1M Nov 26 13:04 ranker.txt


The 4 Candidate source once, on the latest 12 weeks

In [13]:
import gc, time, json
import numpy as np, pandas as pd, lightgbm as lgb, scipy.sparse as sp
from implicit.als import AlternatingLeastSquares

t0 = time.time()
hist = recent[recent["week_ago"].between(0, 11)].copy()      # the latest 12 weeks: predicting next week
hist["t_dat"] = pd.to_datetime(hist["t_dat"])
active = hist["cust"].unique()
print(f"{len(active):,} active customers (bought in the last 12 weeks)")

# 1. Repurchase
rep = (hist.groupby(["cust", "article_id"])
           .agg(last_day=("t_dat", "max"), rep_count=("t_dat", "size"))
           .reset_index()
           .sort_values(["cust", "last_day", "rep_count"], ascending=[True, False, False]))
rep["rep_rank"] = rep.groupby("cust").cumcount()
rep = rep.loc[rep["rep_rank"] < 30, ["cust", "article_id", "rep_rank", "rep_count"]]

# 2. Trending: last week's top 100
top = hist.loc[hist["week_ago"] == 0, "article_id"].value_counts().index[:100].to_numpy()

# 3. ALS: train once, recommend in batches of 50,000 customers
users, items = hist["cust"].unique(), hist["article_id"].unique()
u_idx = pd.Series(np.arange(len(users)), index=users)
i_idx = pd.Series(np.arange(len(items)), index=items)
age = (hist["t_dat"].max() - hist["t_dat"]).dt.days.to_numpy()
mat = sp.csr_matrix(((0.5 ** (age / 28)).astype(np.float32),
                     (u_idx[hist["cust"]].to_numpy(), i_idx[hist["article_id"]].to_numpy())),
                    shape=(len(users), len(items)))
als = AlternatingLeastSquares(factors=64, iterations=15, regularization=0.05, random_state=42)
als.fit(mat, show_progress=False)
als_parts = []
for s in range(0, len(users), 50_000):
    rows = np.arange(s, min(s + 50_000, len(users)))
    ids, scores = als.recommend(rows, mat[rows], N=30, filter_already_liked_items=False)
    als_parts.append(pd.DataFrame({"cust": np.repeat(users[rows], ids.shape[1]),
                                   "article_id": items[ids.ravel()],
                                   "als_rank": np.tile(np.arange(ids.shape[1]), len(rows)),
                                   "als_score": scores.ravel()}))
als_df = pd.concat(als_parts, ignore_index=True)
del als_parts, als, mat; gc.collect()

# 4. Co-purchase: baskets from the last 4 weeks
baskets = hist.loc[hist["week_ago"] <= 3, ["cust", "t_dat", "article_id"]].drop_duplicates()
con.register("b", baskets)
pairs = con.sql("""
    WITH pairs AS (
        SELECT x.article_id AS src, y.article_id AS dst, count(*) AS n
        FROM b x JOIN b y
          ON x.cust = y.cust AND x.t_dat = y.t_dat AND x.article_id <> y.article_id
        GROUP BY 1, 2
    ),
    src_total AS (SELECT article_id AS src, count(*) AS n_src FROM b GROUP BY 1)
    SELECT src, dst, CAST(n AS DOUBLE) / n_src AS p
    FROM pairs JOIN src_total USING (src)
    QUALIFY row_number() OVER (PARTITION BY src ORDER BY n DESC, dst) <= 10
""").df()
con.unregister("b")
last5 = (hist.sort_values("t_dat", ascending=False)
             .drop_duplicates(["cust", "article_id"])
             .groupby("cust").head(5)[["cust", "article_id"]])
cop = (last5.merge(pairs, left_on="article_id", right_on="src")
            .groupby(["cust", "dst"])["p"].sum().rename("cop_score").reset_index()
            .rename(columns={"dst": "article_id"})
            .sort_values(["cust", "cop_score"], ascending=[True, False]))
cop = cop.groupby("cust").head(20)
del baskets, pairs, last5; gc.collect()
print(f"shared pieces ready ({time.time() - t0:.0f} s)")

489,553 active customers (bought in the last 12 weeks)


/usr/local/lib/python3.13/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


shared pieces ready (251 s)


Score Everyone, 40,000 customers at a time

In [14]:
booster = lgb.Booster(model_file=f"{ART}/ranker.txt")
FEATURES = booster.feature_name()
CHUNK = 40_000
recs = []
t0 = time.time()
for i, s in enumerate(range(0, len(active), CHUNK)):
    chunk = active[s:s + CHUNK]
    r, a, c = (d[d["cust"].isin(chunk)] for d in (rep, als_df, cop))
    trend = pd.DataFrame({"cust": np.repeat(chunk, len(top)),
                          "article_id": np.tile(top, len(chunk)),
                          "trend_rank": np.tile(np.arange(len(top)), len(chunk))})

    # same assembly as build_candidates, so the features match training exactly
    cands = pd.concat([d[["cust", "article_id"]] for d in (r, trend, a, c)]).drop_duplicates()
    for d in (r, trend, a, c):
        cands = cands.merge(d, on=["cust", "article_id"], how="left")
    for name, col in (("rep", "rep_rank"), ("trend", "trend_rank"), ("als", "als_rank"), ("cop", "cop_score")):
        cands[f"src_{name}"] = cands[col].notna().astype("int8")
    floats = cands.select_dtypes("float64").columns
    cands[floats] = cands[floats].astype("float32")

    feats = add_features(cands, hist, cust_feat, art)
    for col in FEATURES:
        if feats[col].dtype == "object":
            feats[col] = pd.to_numeric(feats[col], errors="coerce")
    feats["score"] = booster.predict(feats[FEATURES].astype("float32"))
    recs.append(feats.sort_values(["cust", "score"], ascending=[True, False])
                     .groupby("cust").head(12)[["cust", "article_id"]])
    print(f"chunk {i + 1}: {len(chunk):,} customers, {len(cands):,} candidates ({time.time() - t0:.0f} s)")
    del r, a, c, trend, cands, feats; gc.collect()

recs = pd.concat(recs, ignore_index=True)
print(f"{recs['cust'].nunique():,} customers scored, {len(recs):,} recommendations")

chunk 1: 40,000 customers, 5,815,993 candidates (170 s)
chunk 2: 40,000 customers, 5,784,856 candidates (346 s)
chunk 3: 40,000 customers, 5,752,652 candidates (515 s)
chunk 4: 40,000 customers, 5,732,327 candidates (681 s)
chunk 5: 40,000 customers, 5,724,990 candidates (855 s)
chunk 6: 40,000 customers, 5,628,143 candidates (1019 s)
chunk 7: 40,000 customers, 5,496,407 candidates (1178 s)
chunk 8: 40,000 customers, 5,421,796 candidates (1336 s)
chunk 9: 40,000 customers, 5,607,988 candidates (1505 s)
chunk 10: 40,000 customers, 5,588,001 candidates (1665 s)
chunk 11: 40,000 customers, 5,551,548 candidates (1823 s)
chunk 12: 40,000 customers, 5,343,761 candidates (1980 s)
chunk 13: 9,553 customers, 1,256,446 candidates (2015 s)
489,553 customers scored, 5,874,636 recommendations


Save the Recommendations

In [15]:
# Real customer IDs and 10-digit article IDs (H&M article IDs start with 0)
lists = (recs.groupby("cust")["article_id"]
             .apply(lambda s: [str(x).zfill(10) for x in s])
             .rename("article_ids").reset_index())
ids = con.sql("SELECT cust, customer_id FROM cust_map").df()
out = lists.merge(ids, on="cust")[["customer_id", "article_ids"]]
out.to_parquet(f"{ART}/recommendations.parquet", index=False)

cold = {"trending": [str(x).zfill(10) for x in top[:12]],
        "history_end": str(hist["t_dat"].max().date())}
with open(f"{ART}/cold_start.json", "w") as f:
    json.dump(cold, f, indent=2)

print(f"{len(out):,} customers saved; every list has 12 items: {out['article_ids'].apply(len).eq(12).all()}")
out.head()

489,553 customers saved; every list has 12 items: True


,customer_id,article_ids
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,"[0568601043, 0779781015, 0762846031, 091852200..."
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,"[0448509014, 0924243002, 0924243001, 076284603..."
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,"[0794321007, 0765743007, 0924243001, 080500000..."
3,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,"[0896152002, 0730683050, 0791587015, 089615200..."
4,0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...,"[0448509014, 0719530003, 0919365008, 076008400..."


Upload a new version to Kaggle

In [16]:
from pathlib import Path
meta = {"title": "hm-recommender-artifacts",
        "id": f"{KAGGLE_USER}/hm-recommender-artifacts",
        "licenses": [{"name": "CC0-1.0"}]}
(Path(ART) / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

!kaggle datasets version -p {ART} -m "Add batch recommendations and cold-start list"

Starting upload for file recommendations.parquet
100% 38.7M/38.7M [00:01<00:00, 29.9MB/s]
Upload successful: recommendations.parquet (39MB)
Starting upload for file cold_start.json
100% 269/269 [00:00<00:00, 424B/s]
Upload successful: cold_start.json (269B)
Starting upload for file ranker.txt
100% 2.04M/2.04M [00:00<00:00, 2.73MB/s]
Upload successful: ranker.txt (2MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/aparnarameshdatas/hm-recommender-artifacts
